# Lesson 4 : Tools

Microsoft Agent Framework provides a wide variety of built-in tool's object (such as, Code Interpreter, Web Search, File Search, MCP tools, Browser Automation, etc), and you can use these useful tools in your agent.  
With ```FoundryChatClient``` in Microsoft Agent Framework, you can use the following 3 types of tools. :

- **Hosted tools** : As I have mentioned in Lesson 1, ```FoundryChatClient``` uses Azure AI Projects SDK (```azure-ai-projects```) v2. You can work with tools natively hosted in Azure AI Projects SDK. The available tools in this type will vary depending on the type of client. For example, you can use [Claude's web search tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool) when ```AnthropicClient```.
- **Foundry tools** : Microsoft Foundry provides various additional tools in the gallery (catalog) - such as, SharePoint tool, Fabric data agent tool, OpenAPI-integrated tool, or 3rd-party tools, and you can also use these pre-configured additional tools in ```FoundryChatClient```.
- **Tools in Microsoft Agent Framework (MAF)** : The library of Agent Framework SDK also provides native tools. Unlike above server-side tool calling, these native tools are mostly handled as local functions in LLM invocation, and these are processed in Microsoft Agent Framework SDK which runs on your local computers. For example, native MCP tools in Agent Framework (such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, or ```MCPWebsocketTool```) are locally processed in Microsoft Agent Framework along with MCP protocol specification.

In this exercise, we will explore these 3 types of tools with examples of web search tool and MCP tool.

## 1. Hosted tools

### Web Search example

In the first example, we explore web search tool in ```FoundryChatClient``` hosted tools.

Firstly, same as in Lesson 1, we create a client as follows.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Now we create tool definition for native web search tool in Microsoft Foundry, and create an agent with this tool setting.

By calling ```get_web_search_tool()``` method in ```FoundryChatClient```, ```WebSearchPreviewTool``` object (in Azure AI Projects SDK) is internally created and used.

In [2]:
from agent_framework import Agent

web_search_tool = client.get_web_search_tool()
agent = Agent(
    name="WeatherAgentWithSearchTool",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[web_search_tool])

Now we run the agent as follows.

As you see, this agent knows "what day is it today" or "what is the actual weather condition today", because web search is performed internally.

In Lesson 1, the tool execution is run on your local function. In this example, however, Microsoft Foundry will handle this process on server.

In [3]:
from IPython.display import Markdown, display

result = await agent.run("Tell me the weather and temperature in Osaka today.")
display(Markdown(result.text))

Today in **Osaka (Fri, Jan 16, 2026)**: **mostly sunny**.

- **Temperature:** around **15°C / 59°F** high, **~5°C / 41°F** low [Osaka-shi, Osaka, Japan Weather Forecast | AccuWeather](https://www.accuweather.com/en/jp/osaka-shi/225007/weather-forecast/225007)[10 Day Weather - Osaka, Osaka, Japan - The Weather Channel](https://weather.com/weather/tenday/l/Osaka+Osaka+Japan?placeId=441174f51a1951566e6b1d02bd724effab80d7359e5f241540d1aa46dfecc59f)[Osaka, Japan 14 day weather forecast - timeanddate.com](https://www.timeanddate.com/weather/japan/osaka/ext)[Weather - Osaka City - 14-Day Forecast & Rain | Ventusky](https://www.ventusky.com/osaka)[Osaka Weather Forecast](https://www.weather-forecast.com/locations/Osaka/forecasts/latest)

### MCP example

Next we explore MCP tool hosted in Microsoft Foundry client on Microsoft Agent Framework. (This will internally use MCP tool definition in Azure OpenAI Responses API.)

Same as above example, we call built-in ```get_mcp_tool()``` method in ```FoundryChatClient``` to get ```MCPTool``` object.

In this example, we create an agent to answer Microsoft technical questions.<br>
This agent uses a remote MCP server (Streamable HTTP server), which provides information about Microsoft Learn document.

In [4]:
mcp_tool = client.get_mcp_tool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    approval_mode="never_require",
)
agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[mcp_tool],
)

Let's ask a technical question about Microsoft Azure.  
In this call, MCP tool calling (about Microsoft Learn document) is handled in Microsoft Foundry. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [5]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

To create an Azure Storage account with Azure CLI:

1) Sign in (if you’re running CLI locally)
```bash
az login
```

2) Create (or reuse) a resource group
```bash
az group create \
  --name storage-rg \
  --location eastus
```

3) Create the storage account (general-purpose v2)
```bash
az storage account create \
  --name <uniqueStorageAccountName> \
  --resource-group storage-rg \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Notes:
- `<uniqueStorageAccountName>` must be **globally unique** in Azure (3–24 chars, lowercase letters and numbers only).
- Common SKUs: `Standard_LRS`, `Standard_GRS`, `Standard_RAGRS`, `Standard_ZRS`, etc.

Reference: `az storage account create` and the “Create an Azure storage account” guide on Microsoft Learn:  
https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account

## 2. Foundry tools (MCP example)

In the next example, we use tools in Microsoft Foundry.<br>
Foundry tools provides various added values - such as, built-in authentication, toolbox (tool's set for sharing and reuse), guardrails for tool calling, etc. By registering tools on server-side, it also promotes tool reuse within your team.

> Note : Here I don't go details about Foundry toolbox, but **Foundry toolbox** is also an MCP server that has an endpoint URL, so you can handle it in the same way as a regular MCP call using the following ```MCPStreamableHTTPTool```.

In this example, we change above MCP example to use built-in "Microsoft Learn MCP server" tool in Foundry.

### Preparation

Before writing code, please connect to "Microsoft Learn" tool in Microsoft Foundry UI as follows.

1. Open Foundry Portal.
2. Go to "Build" tab.
3. Select "Tools" menu.
4. In "Tools" tab, connect to "Microsoft Learn MCP server" in catalog, and establish connection.

After the connection is established, please **copy the project connection id**.

### Run code

Now let's create an agent with this Foundry tool as follows.<br>
In the following code, **please replace the following ```PROJECT_CONNECTION_ID```** with project connection id that you have obtained above.

> Note : All tools registered in your Foundry project has unique project connection id, and your agents built in Microsoft Agent Framework can then connect to any tools in project by setting this id. (Mostly 3rd party tools in Microsoft Foundry has MCP type.)

In [6]:
# ToDo : fill your below settings
#       (e.g., /subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_RESOURCE_NAME}/projects/{FOUNDRY_PROJECT_NAME}/connections/{CONNECTED_RESOURCE_NAME})
PROJECT_CONNECTION_ID = "xxxxxxxxxx"

agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[
        {
            "type": "mcp",
            "server_label": "mymcp01",
            "server_url": "https://learn.microsoft.com/api/mcp",
            "require_approval": "never",
            "project_connection_id": PROJECT_CONNECTION_ID,
        }
    ],
)

Same as above, let's ask a technical question about Microsoft Azure. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [7]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

Use **az storage account create** (storage accounts are Azure Resource Manager resources, so you’ll typically create/use a resource group first).

```azurecli
# Sign in (skip if already signed in, e.g., Cloud Shell)
az login

# 1) Create a resource group
az group create \
  --name storage-rg \
  --location eastus

# 2) Create a StorageV2 (general-purpose v2) storage account
# Note: storage account name must be globally unique (3-24 lowercase letters/numbers)
az storage account create \
  --name <account-name> \
  --resource-group storage-rg \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

(Optional) Verify:

```azurecli
az storage account show --resource-group storage-rg --name <account-name>
```

Refs:
- https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account
- https://learn.microsoft.com/cli/azure/storage/account?view=azure-cli-latest#az-storage-account-create

## 3. Tools in MAF (MCP example)

Microsoft Agent Framework also provides native built-in MCP tools - such as, ```MCPStdioTool```, ```MCPStreamableHTTPTool```, and ```MCPWebsocketTool```.<br>
Unlike above tools, these built-in tools are processed as local functions (i.e., **client-side tool calling**) in Microsoft Agent Framework SDK, in accordance with MCP protocol specifications.  

This method is useful when custom processing is required for MCP tool calling.<br>
For example, some remote API models might not support MCP STDIO-based server tools, but ```MCPStdioTool``` can do.<br>
If the MCP tool requires some **authentication** token, you can process authentication (display the login screen) in the client and pass the obtained token in the request's header on MCP tool calling, using ```header_provider``` property in ```MCPStreamableHTTPTool```.

In this example, we change above MCP example to use ```MCPStreamableHTTPTool``` in Microsoft Agent Framework (MAF) as follows. (You can verify that MCP tool is being used internally by checking the internal steps by [tracing](./02_trace.ipynb).)

In [8]:
from agent_framework import MCPStreamableHTTPTool

mcp_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP",
    url="https://learn.microsoft.com/api/mcp",
    load_prompts=False,
    approval_mode="never_require",
)

agent = Agent(
    name="MSTechKnowledgeAgent",
    client=client,
    instructions="You are an agent who answers technical questions about Microsoft products and services.",
    tools=[mcp_tool],
)

In [9]:
result = await agent.run("How to create an Azure storage account using Azure CLI ?")
display(Markdown(result.text))

1) Sign in (if you’re running locally):
```bash
az login
```

2) Create a resource group:
```bash
az group create \
  --name storage-resource-group \
  --location eastus
```

3) Create the storage account (general-purpose v2):
```bash
az storage account create \
  --name <account-name> \
  --resource-group storage-resource-group \
  --location eastus \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false
```

Notes:
- `<account-name>` must be **globally unique**, **3–24 characters**, **lowercase letters and numbers only**.
- Choose a different redundancy SKU if needed (for example `Standard_LRS`, `Standard_GRS`, `Standard_ZRS`, etc.).

Reference: https://learn.microsoft.com/azure/storage/common/storage-account-create#create-a-storage-account

## 4. MCP authentication in Foundry tools

In Foundry custom MCP tools, you can configure authentication, written in [here](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/mcp-authentication#supported-authentication-methods).

Especially, when you configure **OAuth identity passthrough**, your client should handle the authentication in your code - such as, obtaining an authentication request, requesting authentication to the user (display the login screen), and invoking authenticated MCP tools.

In this section, we learn how we can process this authentication in Microsoft Agent Framework.

> Note : I won't go into detail here, but using Foundry toolboxs in hosted agent (which is discussed in Lesson 11), you can easily handle OAuth on-behalf-of (OBO) flow for authenticating individual tools without any extra code and settings. (See [here](https://tsmatz.wordpress.com/2026/03/12/microsoft-agent-identity-for-developers/) for OAuth OBO flow with Agent identity.)

### Set up OAuth identity passthrough

First, add custom MCP tools in Microsoft Foundry with OAuth identity passthrough by applying the following steps.

1. Register client application in Entra ID as follows. This client is used for processing login with [OAuth code grant flow](https://tsmatz.wordpress.com/2016/02/24/v2-endpoint-oauth2-client-using-azure-active-directory-and-microsoft-account/).
    - Go to "Entra ID" in Azure Portal.
    - Select "Manage" - "App registrations", and add a new application for multiple tenants.
    - Select "Manage" - "Certificates and Secrets", and create a new secret.
    - **Copy client id (application id) and secret** of this application.
2. Add MCP server and configure OAuth2 as follows.
    - Select "Build" and "Tools" menu in Foundry Portal.
    - In "Tools" tab, connect to "Model Context Protocol (MCP)" in "Custom" category.
    - In a dialog that appears, set as follows.
        - Name: (arbitrary name)
        - Remote MCP Server endpoint: ```https://authclaimview.azurewebsites.net/mcp```
        - Authentication: OAuth Identity Passthrough
        - Client ID: (set above client id)
        - Client secret: (set above secret value)
        - Token URL: ```https://login.microsoftonline.com/common/oauth2/v2.0/token```
        - Auth URL: ```https://login.microsoftonline.com/common/oauth2/v2.0/authorize```
        - Refresh URL: ```https://login.microsoftonline.com/common/oauth2/v2.0/token```
        - Scope: ```openid,offline_access```
    - After submission, a dialog containing redirect URL is displayed. Please **copy this redirect URL**.
    - After adding MCP server, please **copy project conection ID** of this MCP server.
3. Go back to Entra ID application generated above, and set above redirect URL as application's redirect URI. (Select "Web" as platform.)

> Note : The ```offline_access``` scope is required to use refresh token. (In this example, we don't need it.)

For detailed instructions, please see [this official document](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/mcp-authentication).

![OAuth identity passthrough](./assets/foundry_oauth.png)

### Capture a consent request

Now run the following code to connect and handle authentication for MCP tools.<br>
Foundry agent returns a consent request as an event (which type is "```oauth_consent_request```") with a consent URL as follows.

In [10]:
PROJECT_CONNECTION_ID = "<FILL-PROJECT-CONNECTION-ID>"
USER_PROMPT = "Tell me current credential claim information in this session."

# create agent
agent = Agent(
    name="CustomMCPWithAuthAgent",
    client=client,
    instructions="You are a helpful assistant.",
    tools=[
        {
            "type": "mcp",
            "server_label": "authclaimview",
            "server_url": "https://authclaimview.azurewebsites.net/mcp",
            "require_approval": "never",
            "project_connection_id": PROJECT_CONNECTION_ID,
        }
    ],
)

# get OAuth login requests
# (make sure to run with session)
session = agent.create_session()

async def run_agent(s):
    async for chunk in agent.run(
        USER_PROMPT,
        session=s,
        stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
        if chunk.user_input_requests:
            for r in chunk.user_input_requests:
                if r.type == "oauth_consent_request":
                    print("********** consent link **********")
                    print(r.consent_link)
    print("\n")

await run_agent(session)

********** consent link **********
https://logic-apis-japaneast.consent.azure-apim.net/login?data=eyJMb2dpbklkIjoibG9naWMtYXBpcy1qYXBhbmVhc3RfZDkyMmRmMGI3ZmMwNDZjZGE0ZjNjY2I4NDRmMGZmZDRfdG9rZW4iLCJTZXNzaW9uSWQiOiIiLCJMb2dSdW50aW1lUG9saWN5SWQiOm51bGwsIkxvZ0Nvbm5lY3Rpb25JZCI6IjA2MDQzYjU4NjMyYTQ2OWI4ZTdlNTMzY2ZkZTA0OTdkIiwiTG9nQ29ubmVjdG9ySWQiOiJkOTIyZGYwYjdmYzA0NmNkYTRmM2NjYjg0NGYwZmZkNCIsIkxvZ0Vudmlyb25tZW50SWQiOiI0YTI2NzA1MmUxNDg0ZDUyYjBiNTgyNGRjY2NmNjQ5NyIsIkxvZ0FjY291bnROYW1lIjoibG9naWMtYXBpcy1qYXBhbmVhc3QiLCJFeHBpcmF0aW9uVGltZSI6IjIwMjYtMDctMzBUMDk6NDE6MjMuNTYyNDM4MloiLCJEYXRhIjoiR0lxNlZzdkc5TXJQVUFlNHBWVXN2RHpTWXB3RXd2cFF6T2pZQXNBV256TVFBSmM4alUzL0hZRExUb1JmWSt6VXJ6TkF6SWhaTkhRR3lDb3ozQUZvU0gxN3BnMlN5bGRueWIvMklJVDNpeUllWUlhL1RSeENra1NPVWdBOTNRV1c2K0t6bTJlOUJzYUsvdWhoOUpjM1NGcEh3ZmYxSmRhaGY1NXdYUE1JOHd0YXE4dGYxREwxZEs4d2NCZlJGcUJmRXlYeGFBSFFnUnZsMWRZMklhblh0ZnRVR3FtRkx3MGo0VkJCcWdWSFhxRC93WmRKV1JxZzU1UXZHSDJlRzBYZ3c2cHVpa21ZTlFqZ2VTSVNVS3Q5Q3Fpc0s2V21lZmlCLzRYa2RUUEdCQ1VTb0w0NmlVOW

### Perform consent with your web browser

Copy above consent link (URL) and paste it into your web browser's address bar. (Use in-private (incognite) mode.)<br>
This will show the login screen to the user as follows. Please login with Entra ID user and proceed.

![OAuth identity passthrough](./assets/sign_in.png)

![OAuth identity passthrough](./assets/oauth_consent.png)

After the user has logged-in, the token is stored on Foundry platform for the certain period of time, and the window (browser) will automatically close.

### Invoke authenticated tools

After the user has logged-in, run the same request **with the same session**.

By using the same session, ```previous_response_id``` is passed to Foundry agent internally and it recognizes that the user has already logged-in successfully. The agent then invokes MCP tool by including the token (which is obtained during consent) as Authorization HTTP header.<br>
In this example, the tool in this MCP server (```https://authclaimview.azurewebsites.net/mcp```) will return the claim information of the provided token to the agent.

> Note : It might take a while to get the result, because the MCP server warms up.

In [11]:
await run_agent(session)

Current credential claims in this session:

```json
{
  "aud": "00000003-0000-0000-c000-000000000000",
  "iss": "https://sts.windows.net/99880472-cea0-4370-861d-98ea8adf0a8c/",
  "iat": 1785403630,
  "nbf": 1785403630,
  "exp": 1785408502,
  "acct": 0,
  "acr": "1",
  "acrs": ["p1", "urn:user:registersecurityinfo", "pfdr"],
  "amr": ["pwd", "mfa"],
  "app_displayname": "authclaimview-client",
  "appid": "e636c3f5-429b-4cf5-80e5-dd688b985a9a",
  "appidacr": "1",
  "family_name": "Administrator",
  "given_name": "MOD",
  "idtyp": "user",
  "ipaddr": "13.71.146.83",
  "name": "MOD Administrator",
  "oid": "32422ddd-a01d-4e71-bd00-3dd78f92a398",
  "platf": "3",
  "puid": "10032005300780B4",
  "scp": "openid profile email",
  "sid": "006f6b8a-b0f2-eece-9ca7-d3fa08c62940",
  "sub": "aTD8YGm0heIJqd8UweR_liVk9rNJeYEISIQ5HpW8B88",
  "tenant_region_scope": "NA",
  "tid": "99880472-cea0-4370-861d-98ea8adf0a8c",
  "unique_name": "admin@MngEnvMCAP734711.onmicrosoft.com",
  "upn": "admin@MngEnvMCAP7